_Neural Data Science_

Lecturer: Prof. Dr. Philipp Berens, Dr. Jan Lause

Tutors: Jonas Beck, Kyra Kadhim, Jonathan Oesterle, Julius Würzler

Summer term 2026

Student names: <span style='background: yellow'>Frauke von der Haar, Urmi Jana </span>

LLM Disclaimer: <span style='background: yellow'>*Did you use an LLM to solve this exercise? If yes, which one and where did you use it? [Copilot, Claude, ChatGPT, etc.]* </span>

# Neural Data Science Project — Spike Inference

## Inferring spiking activity from calcium imaging data using deep learning

In this project you will train your own deep network to infer neuronal spiking activity from calcium imaging ΔF/F traces. You will work with ground truth data from simultaneous calcium imaging and electrophysiology recordings, preprocess it for training, build and train a neural network, and evaluate your model on held-out neurons.

You are free to use tools, resources and libraries as you see fit. Use comments and markdown cells to document your thought process and to explain your reasoning. We encourage you to compare different algorithms or to implement state of the art solutions. The notebook should be self contained, although you may offload some functions to a `utils.py`. The notebook should be concluded with a final summary / conclusions section.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.signal import resample
from scipy.stats import pearsonr

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

## Context

### Calcium imaging and spike inference

Two-photon calcium imaging records neural activity by measuring fluorescence of genetically encoded calcium indicators (e.g. GCaMP6f, GCaMP6s). When a neuron fires an action potential, calcium flows into the cell, causing the indicator to fluoresce more brightly. The resulting signal — expressed as **ΔF/F** (relative fluorescence change) — is a noisy, temporally smoothed version of the true spiking activity:

- A single action potential produces a fast rise (~50 ms) followed by a slow exponential decay (~200–1000 ms, depending on the indicator)
- The signal is corrupted by shot noise, neuropil contamination, and motion artifacts
- Multiple spikes in quick succession sum nonlinearly

**Spike inference** (also called deconvolution) is the inverse problem of recovering the underlying spike times or spike rates from the observed ΔF/F traces.

### Cascade as a reference

In this project you will train your own neural network for the task of spike inference, similar to what was done in [**Cascade**](https://github.com/HelmchenLabSoftware/Cascade) (Rupprecht et al., *Nature Neuroscience*, 2021). Cascade trains a simple 1D convolutional network (~50k parameters) on a large database of simultaneous calcium imaging and electrophysiology recordings. A key part of their approach is a noise-matched training procedure that resamples the ground truth data to match the noise level and frame rate of any target recording, making the trained models robust across different experimental conditions. Cascade also provides pretrained models for common configurations that can serve as performance baselines.

You can use their code, preprocessing routines, and pretrained models as a reference. The PyTorch implementation is available at [CascadeTorch](https://github.com/PTRRupprecht/CascadeTorch).

**Paper:** Rupprecht P, Carta S, Hoffmann A, Echizen M, Blot A, Kwan AC, Dan Y, Hofer SB, Kitamura K, Helmchen F\*, Friedrich RW\*. *A database and deep learning toolbox for noise-optimized, generalized spike inference from calcium imaging.* Nature Neuroscience (2021). [Link](https://www.nature.com/articles/s41593-021-00895-5)

### The data

The file `spike_inference_data.npz` contains ground truth recordings from simultaneous calcium imaging and electrophysiology. Each recording provides:

| Field | Description |
|---|---|
| `dff` | ΔF/F fluorescence trace (1D float array) |
| `t` | Time vector in seconds (1D float array) |
| `spikes` | Action potential times in seconds (1D float array) |
| `frame_rates` | Imaging frame rate in Hz (scalar) |
| `datasets` | Source dataset identifier (string) |

The data is split into:

- **Training recordings** (`train_*`): Use these to build your preprocessing pipeline and train your network. They come from 4 ground truth datasets recorded with GCaMP6f and GCaMP6s in mouse visual cortex.
- **Test recordings** (`test_*`): A held-out dataset for final evaluation. Do not use these for training.

All recordings were obtained at ~60 Hz and can be resampled to a lower frame rate (e.g. 30 Hz) during preprocessing.

In [ ]:
# load data
def load_data(path="./data"):
    data = np.load(path + "/spike_inference_data.npz", allow_pickle=True)
    return dict(data)

def print_info(data):
    for key in sorted(data.keys()):
        arr = data[key]
        if arr.dtype == object:
            print(f"  [{key:25s}]  {arr.shape}  (variable-length arrays)")
        else:
            print(f"  [{key:25s}]  {arr.shape}  dtype={arr.dtype}")

data = load_data()

print("Overview of the data")
print_info(data)
print(f"\nTraining datasets: {list(data['train_dataset_names'])}")
print(f"Test datasets:     {list(data['test_dataset_names'])}")
print(f"Training recordings: {len(data['train_dff'])}")
print(f"Test recordings:     {len(data['test_dff'])}")

In [ ]:
# Visualize an example training recording
idx = 0  # change this to browse different recordings
dff = data['train_dff'][idx]
t = data['train_t'][idx]
spikes = data['train_spikes'][idx]
fr = data['train_frame_rates'][idx]
ds_name = str(data['train_datasets'][idx])

fig, axes = plt.subplots(2, 1, figsize=(14, 5))

# Full trace
axes[0].plot(t, dff, 'k', linewidth=0.5)
for sp in spikes:
    axes[0].axvline(sp, color='red', alpha=0.3, linewidth=0.5, ymin=0, ymax=0.1)
axes[0].set_ylabel('ΔF/F')
axes[0].set_title(f'Recording {idx}: {ds_name}  |  {fr:.0f} Hz, {len(dff)} frames, '
                  f'{len(spikes)} spikes, {t[-1]:.0f} s')

# Zoomed window
t0, t1 = 30, 50
m = (t >= t0) & (t <= t1)
axes[1].plot(t[m], dff[m], 'k', linewidth=1)
for sp in spikes[(spikes >= t0) & (spikes <= t1)]:
    axes[1].axvline(sp, color='red', alpha=0.6, linewidth=1)
axes[1].set_ylabel('ΔF/F')
axes[1].set_xlabel('Time (s)')
axes[1].set_title(f'Zoomed: {t0}–{t1} s  (red = electrophysiologically recorded spike times)')

plt.tight_layout()
plt.show()

## Research question

<span style='background: yellow'>
Investigating and Mitigating Domain Shift in Deep Learning-Based Spike Inference Across Calcium Indicators
Does spike inference exhibit asymmetric generalization between GCaMP6f and GCaMP6s?
Can mixed-indicator training reduce this domain shift? How does the model architecture affect spike inference performance?
</span>

Some possible directions (pick one, or formulate your own):

1. **Train a spike inference network and evaluate its generalization.** Build a preprocessing pipeline (resampling, noise augmentation, windowing, target smoothing), train a neural net, and evaluate on the held-out test set. How does performance depend on the training data composition and noise level?

2. **Compare architectures.** Implement at least two different network architectures (e.g. the Cascade default, a deeper/shallower variant, one with skip connections or batch normalization). Which design choices matter most for spike inference performance?

3. **How does temporal smoothing of the ground truth affect inference quality?** Train models with different smoothing kernels (e.g. 25, 50, 100, 200 ms) and evaluate the trade-off between temporal precision and robustness. Optionally compare symmetric vs. causal smoothing kernels.

4. **How well does a network trained on one indicator generalize to another?** Train on GCaMP6f data only and test on GCaMP6s (or vice versa). How much does indicator-specific training improve performance?

Implement all steps necessary to answer your question. Think of: preprocessing, model design, training, evaluation, and visualization. The notebook should be concluded with a summary / conclusions section.

# Delete when finished!


In [ ]:

%load_ext autoreload
%autoreload 2

import utils
import warnings
import importlib
%reload_ext autoreload
warnings.warn("Test ")

importlib.reload(utils)

## Preprocessing Pipeline

Every recording goes through the same sequence before a network ever sees it, all implemented in `utils.py`:

test_ recordings will go though this pipeline after network training. 

| Step | What happens | Function |
|---|---|---|
| 1 | Resample every trace onto a uniform 30 Hz grid | `standardize_trace` |
| 2 | Bin continuous spike times into per-frame counts | `bin_discrete_spikes` |
| 2b | Hardware lag correction — evaluated, deliberately skipped | `fit_dataset_level_lag` / `bounded_cross_correlation_alignment` |
| 3 | Remove slow baseline drift from the ΔF/F | `correct_dff_baseline_drift` |
| 4 | Exclude low-SNR / low-spike-count recordings | `robust_quality_control` |
| 5 | Convert discrete spikes to a smoothed Hz-rate target | `smooth_spike_train` |
| 6 | Split recordings into train/validation, stratified by dataset | `partition_recordings` |
| 7 | Match the training-set noise floor across recordings | `augment_train_noise` |
| 8 | Cut continuous traces into fixed-length CNN input windows | `generate_sliding_windows` |
| 9 | Z-score features using train-set statistics 

### Step 1 — Temporal Standardization
`standardize_trace()` · `utils.py`

**Problem.** Raw ΔF/F traces aren't sampled at a perfectly uniform rate, and exact frame rates vary slightly across recordings (~59–61 Hz here). A 1D-CNN needs the same physical duration to be represented by the same number of frames in every recording, or it can't learn one consistent temporal kernel (e.g. for the ~50 ms GCaMP rise time).

**Solution.** Every trace is resampled onto a single uniform 30 Hz grid. When a recording needs downsampling, an 8th-order Chebyshev Type I low-pass filter is applied first (zero-phase, via `sosfiltfilt`), then linearly interpolated onto the exact target grid. If the native rate is already ≤ 30 Hz, filtering is skipped since there's no aliasing risk.

**Why.** Resampling without filtering first lets high-frequency content (shot noise, motion artifacts) fold back down into the calcium-transient band, corrupting the trace permanently.

**Why not.** Cutting off exactly at the new Nyquist frequency would still leak some aliased energy, since Chebyshev filters roll off gradually rather than instantly — hence the small safety margin below it.


In [ ]:
from utils import standardize_trace

TARGET_FS = 30.0  # Hz - global standard for the whole project

phase1_data = {}

print(f"Resampling continuous traces to {TARGET_FS} Hz...")

for idx in range(len(data['train_dff'])):
    raw_dff = data['train_dff'][idx]
    raw_t = data['train_t'][idx]

    # Anti-aliased resampling onto the uniform target grid
    dff_res, t_res = standardize_trace(dff=raw_dff, t=raw_t, target_fs=TARGET_FS)

    # Keep everything needed for the next preprocessing steps together,
    # keyed by recording index so nothing gets misaligned downstream.
    phase1_data[idx] = {
        'dff': dff_res,                              # resampled dF/F trace
        'frame_times': t_res,                        # resampled, uniform time vector
        'spikes': data['train_spikes'][idx],         # RAW spike times (point events - not resampled)
        'dataset': str(data['train_datasets'][idx]),
        'filename': str(data['train_filenames'][idx]),
        'orig_frame_rate': data['train_frame_rates'][idx],
    }

print(f"Successfully resampled {len(phase1_data)} recordings.")
print(f"Recording 0 - Original shape: {data['train_dff'][0].shape} "
      f"(~{data['train_frame_rates'][0]:.1f} Hz)")
print(f"Recording 0 - New shape:      {phase1_data[0]['dff'].shape} ({TARGET_FS} Hz)")


> 📝 **Personal note.** The original version of this function returned `dff_standard,` — a trailing comma that silently turned the return value into a 1-tuple and dropped `t_standard` entirely. Python doesn't complain about that at all;
>  worth double-checking multi-return functions actually unpack the way expected, especially after a refactor.
>
>  — worth remembering *why* you filter before you downsample, not just that you do.

### Step 2 — Discrete Event Binning
`bin_discrete_spikes()` · `utils.py`

**Problem.** Spikes are timestamped continuously (electrophysiology clock); the resampled ΔF/F is discrete (camera frames). Every spike has to be assigned to exactly one frame, with none silently lost or double-counted.

**Solution.** Build bin edges at the midpoints between consecutive frame timestamps (plus extrapolated edges at the two ends), then histogram the spike times into those edges.

**Why.** Midpoint edges give the least-biased assignment when frame intervals aren't perfectly uniform, and they let spike conservation be checked explicitly — the sum of the binned counts must equal the number of spikes that actually fall inside the recording window.

**Why not.** A fixed-width bin (e.g. `[frame, frame + dt)`) is simpler but systematically biases spikes near a boundary toward the earlier frame; midpoint edges center each bin on its own frame instead.


### Step 2b — Hardware Lag Correction: Evaluated and Skipped
`fit_dataset_level_lag()` / `bounded_cross_correlation_alignment()` · `utils.py` — *available, not used here*

**Problem.** Optical and electrical acquisition systems can have a small, fixed synchronization offset. `utils.py` provides two ways to estimate and correct it: a per-recording fit and a more robust per-source-dataset fit.

**Solution.** Skip it — assume negligible hardware lag (0 frames) rather than fitting one from this data.

**Why.** Both available methods estimate the lag by maximizing correlation between ΔF/F and binned spikes, so the result reflects the calcium indicator's own rise/decay kinetics as much as any real delay (synthetic tests: a zero-lag GCaMP6f-like kernel still gets "corrected" by roughly -67 ms, a GCaMP6s-like kernel by roughly -133 ms, from kinetics alone). In this project, every source dataset maps to exactly one indicator (`DS09`/`DS10` = GCaMP6f, `DS14`/`DS15`/`DS16` = GCaMP6s), so a per-*dataset* fit is mathematically identical to a per-*indicator* fit here. Applying either method would bake a different, indicator-correlated time shift into the data before training — for the same reason it's unsafe to fit a lag on the held-out test set: the "correction" would be derived from the very signal whose cross-indicator comparison is the actual research question.

**Why not** *(the alternative, i.e. fitting a lag anyway)*. Even the more robust dataset-level fit doesn't avoid this: pooling within one indicator's recordings reduces recording-to-recording noise, but still converges on that indicator's own kinetics-driven bias, not a true hardware offset. These recordings also appear to come from Cascade's published ground-truth database, where calcium and electrophysiology are synchronized at acquisition — a documented rig constant, if one existed, would be the only safe way to add lag correction back in.


In [ ]:
from utils import bin_discrete_spikes

TARGET_FRAME_RATE = TARGET_FS
 
# bin the spikes intoframes
binned_by_id = {
    rid: bin_discrete_spikes(spike_times=rec['spikes'], frame_times=rec['frame_times'])
    for rid, rec in phase1_data.items()
}

# REMOVED: The dataset grouping and the lag fitting loops have been deleted.

phase2_aligned_data = {}
for rid, rec in phase1_data.items():
    
    phase2_aligned_data[rid] = {
        'aligned_dff': rec['dff'],                 # Passed through unshifted
        'aligned_spike_counts': binned_by_id[rid], # Passed through unshifted
        'optimal_lag_frames': 0,                   # Hardcoded to 0 for pipeline continuity
        'lag_ms': 0.0,                             # Hardcoded to 0 for pipeline continuity
        'dataset': rec['dataset'],
    }

print("Phase 2 complete: Spikes binned. Hardware lag alignment bypassed (assumed pre-aligned).")

> 📝 **Personal note.**
>  consequential decision in the whole pipeline, not a side detail: because every dataset here is a single indicator, *any* correlation-fitted lag — even the "robust" dataset-level one — is really an indicator-level correction, which is exactly the thing the research question is trying to measure, not remove.
>Fell into over engineering trap, with losng sight of the question
> 
> Also worth remembering for the write-up: `bin_discrete_spikes` raises an exception instead of using `assert` for its conservation check, specifically because `assert` statements are stripped out under `python -O` — 

#### Sanity check

Quick visual check that the binned spike counts line up with the ΔF/F rises for one example recording, after alignment.


In [ ]:
# Sanity check: overlay one aligned recording
check_id = 0
rec = phase2_aligned_data[check_id]
t_check = np.arange(len(rec['aligned_dff'])) / TARGET_FRAME_RATE

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(t_check, rec['aligned_dff'], 'k', linewidth=0.7, label='aligned ΔF/F')
spike_frames = np.where(rec['aligned_spike_counts'] > 0)[0]
ax.vlines(t_check[spike_frames], 0, rec['aligned_dff'].max(),
          color='red', alpha=0.4, linewidth=0.8, label='binned spikes')
ax.set_xlim(30, 50)
ax.set_xlabel('Time (s)')
ax.set_ylabel('ΔF/F')
ax.set_title(f"Recording {check_id} ({rec['dataset']}) - lag corrected: "
             f"{rec['optimal_lag_frames']} frames ({rec['lag_ms']:.1f} ms)")
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

### Step 3 — Baseline Drift Correction
`correct_dff_baseline_drift()` · `utils.py`

**Problem.** This dataset only ever provides pre-computed ΔF/F, never raw fluorescence or a neuropil signal — so any slow drift (e.g. residual photobleaching) has to be removed directly from the ΔF/F trace itself, not via the usual neuropil-subtraction-and-rebaseline route.

**Solution.** Estimate the drift as a rolling low percentile (8th) of the ΔF/F trace, smooth that estimate to avoid jagged steps under dense spike bursts, then subtract it.

**Why.** The textbook alternative — dividing by a freshly estimated F0 the way you would for raw fluorescence — assumes the baseline is large and strictly positive. Applied to a trace that's already close to zero (an already-computed ΔF/F), that produces wildly inflated, unstable output.

**Why not.** `compute_dynamic_dff` (neuropil subtraction, then divide by F0) is the right tool when raw fluorescence and neuropil are both available — but this project's data never provides them, so that function has no valid input here.


### Step 4 — Quality Control
`robust_quality_control()` · `utils.py`

**Problem.** Not every recording is usable for training: some have too few spikes to learn from, others are too noisy to trust.

**Solution.** Exclude a recording if it has fewer than 5 total spikes, or if its signal-to-noise ratio falls below 2.5. SNR is estimated the same way Cascade defines it: the 99th-percentile peak minus the median, divided by a frame-rate-normalized noise estimate (median absolute successive difference).

**Why.** Both failure modes actively hurt training — near-empty recordings contribute almost nothing but the zero-target class, and low-SNR recordings teach the network to chase noise instead of real transients.

**Why not.** Plain median absolute deviation from the trace's own median is a simpler noise estimate, but it conflates real calcium transients (which skew the distribution) with actual noise; the successive-difference-based estimate used here is far less sensitive to genuine signal.


In [ ]:
from utils import correct_dff_baseline_drift, robust_quality_control

phase3_clean_data = {}
exclusion_count = 0
for rid, rec in phase2_aligned_data.items():
    dff_clean, drift = correct_dff_baseline_drift(rec['aligned_dff'], fps=TARGET_FRAME_RATE, window_sec=15.0, percentile=8)
    is_valid, qc_metrics = robust_quality_control(dff_clean, rec['aligned_spike_counts'], fps=TARGET_FRAME_RATE, min_spikes=5, snr_threshold=2.5)
    if is_valid:
        phase3_clean_data[rid] = {'dff_clean': dff_clean, 'spikes_binned': rec['aligned_spike_counts'],
                                   'f0_baseline': drift, 'dataset_id': rec['dataset']}
    else:
        exclusion_count += 1
print(f"Retained: {len(phase3_clean_data)}  Excluded: {exclusion_count}")

> 📝 **Personal note.** `robust_quality_control` used to have the same trailing-comma bug as `standardize_trace` — `return is_valid,` — and because a non-empty tuple is always truthy in Python, every recording silently "passed" QC regardless of its actual SNR. Worth flagging: even with the bug fixed, this run still reports 0 recordings excluded out of 83. Probably genuine — all real recordings clear both thresholds — but worth trying one deliberately strict threshold at some point just to confirm QC is actually capable of excluding something, not still silently vacuous some other way.


### Step 5 — Target Smoothing
`smooth_spike_train()` · `utils.py`

**Problem.** Training against a strictly discrete spike-count target gives an extremely jagged loss landscape: a one-frame timing error (~33 ms at 30 Hz) is penalized as a complete miss, leaving the network no gradient signal toward "close."

**Solution.** Convolve the binned spike counts with a Gaussian kernel ($\sigma$ = 0.05 s, matching Cascade's own default at this frame rate) to turn discrete counts into a continuous rate target, in Hz.

**Why.** A smoothed, continuous target gives proportional gradients for near-miss predictions, so the network can learn the indicator's rise/decay kinetics instead of getting stuck matching exact frame boundaries.

**Why not.** A narrower kernel preserves more temporal precision but keeps more of the discreteness problem; a much wider one is easier to fit but blurs away exactly the timing information spike inference is trying to recover.


In [ ]:

# TARGET PARAMETERS (Cascade Standard)

SIGMA_SECONDS = 0.05

print(f"Executing Phase 4: Target Label Smoothing (sigma={SIGMA_SECONDS}s)...")

# 1. Apply smoothing strictly PER RECORDING to prevent boundary leakage
for recording_id, rec_data in phase3_clean_data.items(): # FIX: Iterate over phase3_clean_data
    
    # Extract the discrete aligned bins you generated in Phase 3
    binned_spikes = rec_data['spikes_binned'] # FIX: Update to the correct Phase 3 key
    
    # Smooth the spikes
    smoothed_rates = utils.smooth_spike_train(
        spike_counts=binned_spikes,
        target_fs=TARGET_FRAME_RATE,
        sigma_sec=SIGMA_SECONDS
    )
    
    # Store the continuous targets back into your master dictionary
    phase3_clean_data[recording_id]['smoothed_spike_rates'] = smoothed_rates # FIX: Save to phase3_clean_data

print("Phase 4 smoothing complete. Continuous targets added to dictionary.")

> 📝 **Personal note.** This function used to return the smoothed counts *per frame*, not per second, despite being labeled "Hz" everywhere — docstring, plot axis, variable names. `gaussian_filter1d` uses a mass-preserving kernel, so summing its output recovers the original spike count, not that count scaled by frame rate. Harmless as long as `target_fs` is constant everywhere in the pipeline, but would silently break the moment frame rates are mixed or compared against another Hz-scaled source. Good reminder to check that units aren't just labeled correctly, but actually computed that way.


## Sanity check 

In [ ]:
# 2. Visual Sanity Check
# Dynamically select the first recording in your dictionary for visualization
example_rec_id = list(phase3_clean_data.keys())[0]
example_data = phase3_clean_data[example_rec_id]

# Slice the data for a 10-second plot
frames_to_plot = int(10 * TARGET_FRAME_RATE)
time_axis = np.arange(frames_to_plot) / TARGET_FRAME_RATE

# FIX: Request 'spikes_binned' instead of the outdated 'aligned_spike_counts'
binned_plot = example_data['spikes_binned'][:frames_to_plot] 
smoothed_plot = example_data['smoothed_spike_rates'][:frames_to_plot]

# Generate Publication-Ready Plot
fig, ax1 = plt.subplots(figsize=(10,3), dpi=120)

# Plot discrete spikes as a stem plot on a secondary y-axis for clarity
ax2 = ax1.twinx()
ax2.stem(time_axis, binned_plot,
         linefmt='grey', markerfmt='k1', basefmt=' ', label='Discrete Bins')
ax2.set_ylabel('Discrete Count', color='grey')
ax2.set_ylim(0, max(binned_plot) + 1)
ax2.tick_params(axis='y', labelcolor='grey')

# Plot the smoothed continuous target
ax1.plot(time_axis, smoothed_plot,
         color='blue', linewidth=2, label=rf'Smoothed Rate ($\sigma={SIGMA_SECONDS}$s)')
ax1.set_xlabel('Time (seconds)')
ax1.set_ylabel('Spike Rate (Hz)', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

plt.title('Phase 4: Transformation of Discrete Bins to Continuous Target')
fig.tight_layout()
plt.show()

### Step 6 — Train / Validation Partitioning
`partition_recordings()` · `utils.py`

**Problem.** Splitting at the level of individual sliding windows would leak near-duplicate, heavily overlapping windows across the train/validation boundary, inflating validation performance. The split has to happen at the whole-recording level, before any windowing.

**Solution.** Randomly assign whole recordings to train or validation (15%), stratified by source dataset — which in this project means stratified by indicator, since each dataset is a single indicator.

**Why.** A plain pooled-random split over only 83 recordings could easily land a validation set skewed toward one indicator by chance. For a research question specifically about GCaMP6f-vs-6s asymmetry, that would make a genuine generalization gap indistinguishable from validation-set sampling noise.

**Why not.** An unstratified split is simpler and perfectly fine when indicator identity isn't the question being asked — it just isn't safe to use here.


### Step 7 — Noise Augmentation
`augment_train_noise()` · `utils.py`

**Problem.** A CNN trained only on clean recordings learns to key on high-frequency texture that doesn't exist in noisier data, producing false positives the moment it sees a genuinely noisy recording. This matters here specifically: if GCaMP6f and GCaMP6s recordings don't just differ in kinetics but also happen to differ in typical noise level, an apparent "domain shift" between indicators could really just be a noise-level shift in disguise.

**Solution.** Find the worst (highest) per-recording noise level in the training set, then for every cleaner recording, add Gaussian noise calibrated to bring it up to that same floor — keeping both the original clean copy and the noise-matched copy (doubling the training set).

**Why.** Training against a uniform, worst-case noise floor forces the network to rely on features that survive at every noise level actually present, rather than shortcuts that only work on the cleanest recordings.

**Why not.** Training only on the noise-augmented copies, and dropping the clean originals, would throw away real usable signal for no benefit; keeping both preserves the clean data while still exposing the network to the harder noise regime.


### Step 8 — Sliding Window Generation
`generate_sliding_windows()` · `utils.py`

**Problem.** A 1D-CNN needs fixed-length input; the recordings themselves are long, continuous traces of very different lengths.

**Solution.** Slide a 64-frame window (~2.1 s at 30 Hz) across each recording, pairing every window with its (approximately) center-aligned smoothed spike rate; stride 2 to reduce temporal autocorrelation between adjacent windows.

**Why.** 64 frames covers several times the GCaMP rise/decay time, giving the network enough temporal context to resolve individual transients without windows so long that training examples become mostly redundant with their neighbors.

**Why not.** Stride 1 would use more of the available data per recording, but adjacent windows would then differ by a single frame, making validation performance overly optimistic about how much genuinely new information it's seeing; stride 2 trades a little data for less redundant sampling.


In [ ]:
# We use a window size of 64 frames (~2.13 seconds at the TARGET_FRAME_RATE of 30 Hz)
WINDOW_SIZE = 64  

print("Executing Phase 5: Macro-Level Partitioning & Sliding Windows...")

# 1. Macro-level partitioning of the processed data into Train and Validation sets
# Note: The 'test_*' recordings are already natively held out in the raw data dict.
train_recordings, val_recordings = utils.partition_recordings(
    dataset=phase3_clean_data, 
    val_ratio=0.15, 
    seed=42
)

print(f"Strict Partition Complete: {len(train_recordings)} training and {len(val_recordings)} validation recordings.")

# ---------------------------------------------------------
# NEW: DOMAIN RANDOMIZATION (NOISE MATCHING)
# We ONLY augment the training set to prevent data leakage!
# ---------------------------------------------------------
print(f"Original training recordings: {len(train_recordings)}")
train_recordings = utils.augment_train_noise(train_recordings, target_fs=TARGET_FRAME_RATE)
print(f"Augmented training recordings (Clean + Noisy clones): {len(train_recordings)}")


# 2. Micro-level sliding window generation
# (Using stride=2 to prevent temporal autocorrelation)
X_train, y_train = utils.generate_sliding_windows(
    isolated_dataset=train_recordings, 
    window_size=WINDOW_SIZE,
    stride=2
)

X_val, y_val = utils.generate_sliding_windows(
    isolated_dataset=val_recordings, 
    window_size=WINDOW_SIZE,
    stride=2
)

print("-" * 50)
print(f"Generated Sliding Windows (Window Size: {WINDOW_SIZE} frames):")
print(f"Training Matrices   -> X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"Validation Matrices -> X_val:   {X_val.shape}, y_val:   {y_val.shape}")
print("-" * 50)

> 📝 **Personal note.** Started wondering whether an apparent 6f-vs-6s "domain shift" could really just be a noise-level difference between the two indicators' recordings rather than a kinetics difference — that's exactly what Step 7's noise-matching is meant to guard against, so it's worth reporting the per-recording noise levels inside `augment_train_noise`, split by indicator, as an explicit check rather than just trusting that augmentation handled it. Also: `generate_sliding_windows` used to silently read the wrong dictionary keys (`aligned_dff`/`aligned_spikes`, the Phase-2 fields) no matter what was passed in, so it kept training on pre-QC, pre-baseline-correction data even after "fixing" QC — a good reminder that "the code ran without an error" doesn't mean it used the data you think it did.


In [ ]:
### Step 9 — Feature Scaling
`scale_features()` · `utils.py`

**Problem.** Networks train more reliably on standardized input, but computing normalization statistics from the validation set too would leak information about it into preprocessing.

**Solution.** Compute one global mean and standard deviation from the training windows only, then apply that exact same transform to both train and validation.

**Why.** Fitting the scaler only on training data keeps validation genuinely held out; using a single global scalar (not per-timestep) preserves the temporal shape of the calcium transient instead of flattening it.

**Why not.** Per-timestep normalization is the default in some pipelines, but here it would normalize each position in the window independently — destroying exactly the rise/decay shape the network is supposed to learn from.


In [ ]:
from utils import scale_features

X_train_scaled, X_val_scaled = scale_features(X_train, X_val)

print(f"Scaled X_train shape: {X_train_scaled.shape}")
print(f"Scaled X_val shape: {X_val_scaled.shape}")

> 📝 **Personal note.** Easy mistake to make here is normalizing per-timestep instead of with a single global scalar — it "looks" more careful, but for a temporal window it actively erases the signal shape you're trying to preserve. Worth double-checking this stays the global-scalar version if this function ever gets revisited.


### Saving preprocessed data

In [ ]:
from datetime import datetime
import os


save_dir = "data/preprocessed"
os.makedirs(save_dir, exist_ok=True)

current_date = datetime.now().strftime("%Y-%m-%d")

filename = f"train_preprocessed_spike_data_{current_date}.npz"
full_path = os.path.join(save_dir, filename)

np.savez(
    full_path,
    X_train=X_train_scaled, 
    X_val=X_val_scaled,            
)

print(f"Preprocessing complete. Data securely saved to: {full_path}")

## model design 

placeholder

In [ ]:

import torch
from torch.utils.data import TensorDataset, DataLoader

print("--- Converting to PyTorch Tensors ---")

# 1. Convert the SCALED features and targets to PyTorch Tensors
tensor_X_train = torch.tensor(X_train_scaled, dtype=torch.float32)
# We need to add a channel dimension for 1D-CNNs: (Batch, Channels, Length)
tensor_X_train = tensor_X_train.unsqueeze(1) 
tensor_y_train = torch.tensor(y_train, dtype=torch.float32)

tensor_X_val = torch.tensor(X_val_scaled, dtype=torch.float32)
tensor_X_val = tensor_X_val.unsqueeze(1)
tensor_y_val = torch.tensor(y_val, dtype=torch.float32)

# 2. Create Datasets
train_dataset = TensorDataset(tensor_X_train, tensor_y_train)
val_dataset = TensorDataset(tensor_X_val, tensor_y_val)

# 3. Create DataLoaders
# CRITICAL: shuffle=True fixes the temporal autocorrelation problem!
BATCH_SIZE = 256
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train Loader: {len(train_loader)} batches of size {BATCH_SIZE}")
print(f"Val Loader: {len(val_loader)} batches of size {BATCH_SIZE}")

## training

#### train: 6f only


### train: 6s only

### train: mixed

## evaluation

In [ ]:

import torch
import numpy as np

# Ensure your model is in evaluation mode (turns off dropout/batchnorm)
# model.eval() 

print("--- Evaluating Model on Validation Set ---")

# We don't need gradients for evaluation, so we use torch.no_grad() to save memory
with torch.no_grad():
    # Pass the validation data through the network
    # (Assuming your validation tensors are loaded and on the correct device)
    # val_predictions = model(tensor_X_val.to(device)).squeeze()
    
    # FOR NOW: Let's create dummy predictions just to test the pipeline!
    # Delete these two lines once you have a real trained `model`
    print("[Note: Using dummy predictions to test the metric pipeline]")
    val_predictions = tensor_y_val + torch.normal(mean=0.0, std=0.2, size=tensor_y_val.shape)

    # Move predictions back to CPU and convert to NumPy
    y_pred_np = val_predictions.cpu().numpy()
    y_true_np = tensor_y_val.cpu().numpy()

# Call our new rigorous evaluation function
# You can tweak the threshold later by plotting a Precision-Recall curve!
metrics = utils.evaluate_predictions(y_true=y_true_np, y_pred=y_pred_np, threshold=0.1)

print(f"Mean Squared Error (MSE): {metrics['mse']:.4f}")
print(f"Pearson Correlation (r):  {metrics['pearson_r']:.4f}  <-- How well timings match")
print(f"Detection F1-Score:       {metrics['f1_score']:.4f}  <-- How well spikes were found")

## visualisation

# summary / conclusion